# 第13章　実装ガイド：分類 ― EfficientNetで最初から最後まで

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 13.1　データを読み込む

In [ ]:
import torch, timm
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 事前学習(ImageNet)の重みを使うので、学習時と“同じ”ImageNetのmean/stdで正規化する（省くと精度が落ちる）
norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),          # 左右反転は部位で可否が変わる（眼底・四肢は可、腹部CTは不可 ― データ拡張の章）
    transforms.RandomRotation(10),
    transforms.ToTensor(), norm,
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), norm,
])

# フォルダ名がそのままクラスになる（入門編の「やってみる」の章の眼底なら dr/train/DR, dr/train/No_DR）
train_ds = datasets.ImageFolder("data/train", transform=train_tf)
val_ds   = datasets.ImageFolder("data/val",   transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=4)

## 13.2　モデル・損失・最適化を用意する

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = timm.create_model("efficientnet_b0", pretrained=True,
                          num_classes=len(train_ds.classes)).to(device)

# 少数クラスほど重くする（枚数の逆数）。クラス数が変わっても自動で追従する
counts = torch.bincount(torch.tensor(train_ds.targets), minlength=len(train_ds.classes)).float()
weights = (counts.sum() / (len(train_ds.classes) * counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

## 層別学習率を実装する ― 出力層は大きく、学習済みの部分は小さく

In [ ]:
def split_head_body(model):
    """timmのモデルを、付け替えた出力層(head)と事前学習済みの本体(body)に分ける。"""
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]
    body = [p for p in model.parameters() if id(p) not in head_ids]
    return head, body

head, body = split_head_body(model)

optimizer = torch.optim.AdamW(
    [{"params": body, "lr": 1e-5},     # 学習済みの部分：小さく（出力層の1/10〜1/100が目安）
     {"params": head, "lr": 1e-3}],    # 出力層：大きく（乱数から始まるため）
    weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

## 2段構えで学習する ― 特徴抽出 → ファインチューニング

In [ ]:
# ── 第1段：特徴抽出（学習済みの重みを凍結し、出力層だけを学習する）
for p in body:
    p.requires_grad = False
opt1 = torch.optim.AdamW(head, lr=1e-3, weight_decay=1e-4)
#   この opt1 で、13.3の学習ループを数エポックだけ回す

# ── 第2段：ファインチューニング（凍結を解き、層別学習率で全体を動かす）
for p in body:
    p.requires_grad = True
opt2 = torch.optim.AdamW(
    [{"params": body, "lr": 1e-5},
     {"params": head, "lr": 1e-3}],
    weight_decay=1e-4)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=30)
#   スケジューラは最適化器に結び付いている。最適化器を作り直したら、必ず作り直す

In [ ]:
for m in model.modules():
    if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
        m.eval()          # model.train() の“後”に呼ぶ（train()が全体をtrainに戻すため）

## 13.3　学習ループ ― 検証で最良を保存する

In [ ]:
from sklearn.metrics import average_precision_score, recall_score
POS = train_ds.class_to_idx["DR"]   # 陽性クラスは「名前」で引く（"DR" は自分のデータの陽性フォルダ名に）

SELECT_KEY = "ap"    # 何で選ぶかを、コードの外から見える名前で固定する

def evaluate(model, loader, device):
    """本書で共通の契約：(model, loader, device) を受け取り、指標の辞書を返す。
    第22章の学習ループもこの契約を前提にする。戻り値をタプルにすると呼び先が壊れる。"""
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            prob = torch.softmax(model(x.to(device)), dim=1)   # 確率に直す
            ps += prob[:, POS].cpu().tolist(); ys += y.tolist()  # 陽性クラスの確率
    y_bin = [1 if v == POS else 0 for v in ys]                # 陽性クラス vs それ以外
    n, n_pos = len(y_bin), sum(y_bin)
    # 本例は陽性と陰性を区別するモデルの選択用評価。片方が無ければ保存に進まない。
    if n == 0 or n_pos == 0 or n_pos == n:
        raise ValueError(f"選択用の検証データに陽性と陰性が必要です（陽性{n_pos}/{n}例）")
    if not all(0.0 <= p <= 1.0 for p in ps):
        raise ValueError("予測確率に非有限値または範囲外の値があります")
    pred = [1 if p >= 0.5 else 0 for p in ps]
    return {"ap": average_precision_score(y_bin, ps),
            "sens": recall_score(y_bin, pred),              # 参考：閾値0.5での感度
            "n_pos": n_pos, "n": n}

best = float("-inf")
for epoch in range(30):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
    scheduler.step()
    m = evaluate(model, val_loader, device)         # 選択はAP、感度は参考
    if m[SELECT_KEY] > best:                        # 選ぶ基準は感度ではなくAP
        best = m[SELECT_KEY]
        torch.save(model.state_dict(), "best_model.pth")
    print(f"epoch {epoch}: val AP = {m['ap']:.3f}  参考:感度(0.5) = {m['sens']:.3f}"
          f"  (陽性{m['n_pos']}/{m['n']}例)")

## 13.4　推論する

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()
img = val_tf(Image.open("new_case.png").convert("RGB")).unsqueeze(0).to(device)
with torch.no_grad():
    prob = model(img).softmax(1)
pred = prob.argmax(1).item()
print(f"予測: {train_ds.classes[pred]}（確信度 {prob.max().item():.2f}）")

## 学習ログを読む ― 出力例の読み方

```text
epoch 0:  train_loss=0.689  val AP=0.61  sens=0.58
epoch 4:  train_loss=0.402  val AP=0.78  sens=0.74
epoch 9:  train_loss=0.231  val AP=0.86  sens=0.83
epoch 14: train_loss=0.108  val AP=0.88  sens=0.85   ← 選択用APが最大。ここが best_model.pth に保存される
epoch 19: train_loss=0.041  val AP=0.86  sens=0.83
epoch 24: train_loss=0.012  val AP=0.83  sens=0.81   ← この例では学習損失は下がり続け、検証指標は下降
```

## 確率を正直にする ― 温度スケーリングによる較正（キャリブレーション）

In [ ]:
import torch, torch.nn as nn

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()          # 基底クラス(nn.Module)側の初期化を先に済ませる。省くと層が登録されない
        self.log_T = nn.Parameter(torch.zeros(1))      # T=exp(log_T)。正を保証
    def forward(self, logits):
        return logits / self.log_T.exp()

def fit_temperature(val_logits, val_labels):           # 検証データのロジットと正解
    # 入力は計算グラフから切り離し、scaler と同じ device（CPU）へ揃える。CUDA上のロジットを
    # そのまま渡すと device 不一致、勾配付きのまま渡すと LBFGS の反復で再逆伝播の問題が起きる
    val_logits = val_logits.detach().float().cpu()
    val_labels = val_labels.detach().long().cpu()
    scaler = TemperatureScaler()
    opt = torch.optim.LBFGS(scaler.parameters(), lr=0.01, max_iter=50)
    nll = nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad()
        loss = nll(scaler(val_logits), val_labels)     # 温度だけを動かしてNLLを最小化
        loss.backward(); return loss
    opt.step(closure)
    return scaler.log_T.exp().item()                   # 較正された温度T

## 較正を目で見る ― 信頼性曲線（reliability diagram）とECEをコードで描く

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# 検証で集めた y_true, y_prob を使う。ここでは「自信過剰」なモデルを合成して再現する
rng = np.random.default_rng(0)
# ① まず正しく較正された確率 p_cal を作り、その確率どおりに正解を生成する
p_cal  = rng.uniform(0.0, 1.0, 2000)
y_true = (rng.uniform(0.0, 1.0, 2000) < p_cal).astype(int)
# ② 自信過剰なモデル＝ロジットを鋭くして確率を0/1側へ押し出す（額面より極端な確信度）
logit  = np.log(p_cal / (1.0 - p_cal))
y_prob = 1.0 / (1.0 + np.exp(-logit / 0.5))     # T=0.5で鋭化＝自信過剰

frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfectly calibrated")
plt.plot(mean_pred, frac_pos, "o-", label="model")
plt.xlabel("mean predicted probability"); plt.ylabel("fraction of positives")
plt.title("Reliability diagram"); plt.legend(); plt.show()

In [ ]:
def ece(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, edges) - 1, 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        mean_prob = y_prob[m].mean()   # その帯でモデルが言った平均陽性確率
        frac_pos  = y_true[m].mean()   # その帯で実際に陽性だった割合（正解率ではない）
        total += m.mean() * abs(mean_prob - frac_pos)   # 件数で重み付け
    return total

# 温度スケーリング：推定ロジットを T>1 で割って確信度を和らげる（ここは T=2 で②の鋭化を戻す）
p_ = np.clip(y_prob, 1e-6, 1.0 - 1e-6)
y_prob_T = 1.0 / (1.0 + np.exp(-np.log(p_ / (1.0 - p_)) / 2.0))
print(f"ECE(補正前) = {ece(y_true, y_prob):.3f}  →  ECE(T=2) = {ece(y_true, y_prob_T):.3f}")   # 0に近いほど較正が良い